# Preprocessing — Student Depression Dataset

Apply manual ordinal/binary encodings, group rare cities, build the sklearn `ColumnTransformer`, split into train/test, and persist the splits for `03_modeling.ipynb`.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data_loader import load_raw
from src.preprocessing import prepare_features, build_preprocessor

RANDOM_STATE = 42
df = load_raw()
df.shape

(27901, 18)

## 1. Manual encodings

In [2]:
X, y = prepare_features(df)
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))
X.head()

X shape: (27901, 16)
y distribution:
Depression
1    0.585499
0    0.414501
Name: proportion, dtype: float64


,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,0.0,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5.5,2.0,B.Pharm,1.0,3.0,1.0,0.0
1,1.0,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5.5,1.0,BSc,0.0,3.0,2.0,1.0
2,0.0,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,4.5,2.0,BA,0.0,9.0,1.0,1.0
3,1.0,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7.5,1.0,BCA,1.0,4.0,5.0,1.0
4,1.0,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5.5,1.0,M.Tech,1.0,1.0,1.0,0.0


In [3]:
encoded_cols = ["Gender", "Sleep Duration", "Dietary Habits",
                "Have you ever had suicidal thoughts ?",
                "Family History of Mental Illness"]
X[encoded_cols].describe()

,Gender,Sleep Duration,Dietary Habits,Have you ever had suicidal thoughts ?,Family History of Mental Illness
count,27901.000000,27883.000000,27889.000000,27901.000000,27901.000000
mean,0.442780,6.379174,0.904407,0.632809,0.483961
std,0.496724,1.590558,0.796965,0.482048,0.499752
min,0.000000,4.500000,0.000000,0.000000,0.000000
25%,0.000000,4.500000,0.000000,0.000000,0.000000
50%,0.000000,5.500000,1.000000,1.000000,0.000000
75%,1.000000,7.500000,2.000000,1.000000,1.000000
max,1.000000,8.500000,2.000000,1.000000,1.000000


In [4]:
print("Unique cities after grouping:", X["City"].nunique())
print(X["City"].value_counts().head(10))
print("\n'Other' count:", (X["City"] == "Other").sum())

Unique cities after grouping: 31
City
Kalyan         1570
Srinagar       1372
Hyderabad      1340
Vasai-Virar    1290
Lucknow        1155
Thane          1139
Ludhiana       1111
Agra           1094
Surat          1078
Kolkata        1066
Name: count, dtype: int64

'Other' count: 26


## 2. Train/test split (stratified)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train target share:", y_train.mean())
print("Test target share:", y_test.mean())

Train: (22320, 16) | Test: (5581, 16)
Train target share: 0.5854838709677419
Test target share: 0.5855581437018456


## 3. Build and inspect the preprocessor

In [6]:
preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train, y_train)
X_test_t = preprocessor.transform(X_test)
print("Transformed train shape:", X_train_t.shape)
print("Transformed test shape:", X_test_t.shape)

Transformed train shape: (22320, 82)
Transformed test shape: (5581, 82)


/home/kuba/Projects/Student-Depression-Analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [7]:
feature_names = preprocessor.get_feature_names_out()
print("Number of features after preprocessing:", len(feature_names))
feature_names[:20]

Number of features after preprocessing: 82


array(['num__Gender', 'num__Age', 'num__Academic Pressure',
       'num__Work Pressure', 'num__CGPA', 'num__Study Satisfaction',
       'num__Job Satisfaction', 'num__Sleep Duration',
       'num__Dietary Habits',
       'num__Have you ever had suicidal thoughts ?',
       'num__Work/Study Hours', 'num__Financial Stress',
       'num__Family History of Mental Illness', 'cat__City_Ahmedabad',
       'cat__City_Bangalore', 'cat__City_Bhopal', 'cat__City_Chennai',
       'cat__City_Delhi', 'cat__City_Faridabad', 'cat__City_Ghaziabad'],
      dtype=object)

## 4. Persist the splits

We save the pre-transform `X` so that the modeling notebook can wrap the `preprocessor` inside a `Pipeline(preprocessor, model)`. That keeps the preprocessor inside cross-validation and prevents leakage.

In [8]:
from pathlib import Path
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(out_dir / "X_train.parquet")
X_test.to_parquet(out_dir / "X_test.parquet")
y_train.to_frame("Depression").to_parquet(out_dir / "y_train.parquet")
y_test.to_frame("Depression").to_parquet(out_dir / "y_test.parquet")
print("Saved to", out_dir.resolve())

Saved to /home/kuba/Projects/Student-Depression-Analysis/data/processed
